In [1]:
import pandas as pd
import pickle
import os
import re
import json
from collections import defaultdict
from tqdm.auto import tqdm
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.regex_extractor import extract_volume_percent

TRAIN_BIO_PATH = "../../data/processed/train_bio.csv"
CATALOG_PATH = "../../data/external/products_5ka_raw.json"
OUTPUT_LIBRARY_PATH = "../../data/external/hybrid_library.pkl"

print("--- Шаг 1: Загрузка исходных данных ---")
try:
    df_train = pd.read_csv(TRAIN_BIO_PATH, sep=";")
    df_train["tokens"] = df_train["tokens"].apply(eval)
    df_train["tags"] = df_train["tags"].apply(eval)
    print(f"Загружено {len(df_train)} записей из train_bio.csv")

    with open(CATALOG_PATH, 'r', encoding='utf-8') as f:
        catalog_data = json.load(f)
    print(f"Загружено {len(catalog_data)} товаров из каталога.")

except FileNotFoundError as e:
    print(f"Ошибка: Не найден необходимый файл. {e}")
    df_train = None
    catalog_data = None

print("\n--- Инициализация завершена ---")

--- Шаг 1: Загрузка исходных данных ---
Загружено 23163 записей из train_bio.csv
Загружено 13903 товаров из каталога.

--- Инициализация завершена ---


In [2]:
def extract_entities_from_bio(df: pd.DataFrame, entity_label: str) -> list:
    entities = set()
    for _, row in df.iterrows():
        tokens = row['tokens']
        tags = row['tags']

        i = 0
        while i < len(tags):
            if tags[i] == f"B-{entity_label}":
                current_entity_tokens = [tokens[i]]
                j = i + 1
                while j < len(tags) and tags[j] == f"I-{entity_label}":
                    current_entity_tokens.append(tokens[j])
                    j += 1

                entities.add(" ".join(current_entity_tokens))
                i = j
            else:
                i += 1

    return sorted(list(entities), key=len, reverse=True)

print("--- Шаг 2: Извлечение 'золотых' списков TYPE и BRAND из train.csv ---")
if df_train is not None:
    golden_types = extract_entities_from_bio(df_train, "TYPE")
    golden_brands = extract_entities_from_bio(df_train, "BRAND")
    print(f"Извлечено {len(golden_types)} уникальных сущностей TYPE.")
    print(f"Извлечено {len(golden_brands)} уникальных сущностей BRAND.")
    print("\nПример 5 самых длинных брендов:")
    print(golden_brands[:5])
else:
    print("Ошибка: обучающие данные не были загружены.")
    golden_types, golden_brands = [], []

--- Шаг 2: Извлечение 'золотых' списков TYPE и BRAND из train.csv ---
Извлечено 13648 уникальных сущностей TYPE.
Извлечено 3748 уникальных сущностей BRAND.

Пример 5 самых длинных брендов:
['▁а . рост а гро ком п лек', '▁а . рост а гро комплекс', "▁ar khan gel ' sk kh leb", '▁б . ю . а лек сандр ов', "▁ar khan gel ' s kk hle"]


In [3]:
print("\n--- Шаг 3: Создание гибридной контекстной библиотеки ---")
hybrid_library = defaultdict(lambda: defaultdict(set))

if df_train is not None:
    for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="Обработка train.csv"):
        entities_in_row = defaultdict(list)
        for token, tag in zip(row['tokens'], row['tags']):
            if tag != 'O':
                entities_in_row[tag[2:]].append(token)

        if 'TYPE' in entities_in_row:
            main_type = entities_in_row['TYPE'][0].lower()
            if 'BRAND' in entities_in_row:
                hybrid_library[main_type]['brands'].update(entities_in_row['BRAND'])
            if 'VOLUME' in entities_in_row:
                hybrid_library[main_type]['volumes'].update(entities_in_row['VOLUME'])
            if 'PERCENT' in entities_in_row:
                hybrid_library[main_type]['percents'].update(entities_in_row['PERCENT'])

print(f"Обработано {len(df_train)} записей из train.csv. Найдено {len(hybrid_library)} контекстов.")

if catalog_data:
    for item in tqdm(catalog_data, desc="Обработка каталога"):
        name = item.get("name", "")
        weight = item.get("weight", "")
        category_name = item.get("category_name", "")

        if not category_name or not isinstance(category_name, str):
            continue

        atomic_types = {p.strip() for p in re.split(r'[,/&]| и ', category_name.lower()) if p.strip() and len(p.strip()) > 2}

        full_text_to_scan = f"{name} {weight}"
        vp_entities = extract_volume_percent(full_text_to_scan)

        for start, end, label in vp_entities:
            entity_text = full_text_to_scan[start:end]
            entity_type = 'volumes' if label == 'B-VOLUME' else 'percents'

            for t in atomic_types:
                hybrid_library[t][entity_type].add(entity_text)

print(f"Обработано {len(catalog_data)} товаров из каталога. Итоговое количество контекстов: {len(hybrid_library)}.")


--- Шаг 3: Создание гибридной контекстной библиотеки ---


Обработка train.csv:   0%|          | 0/23163 [00:00<?, ?it/s]

Обработано 23163 записей из train.csv. Найдено 349 контекстов.


Обработка каталога:   0%|          | 0/13903 [00:00<?, ?it/s]

Обработано 13903 товаров из каталога. Итоговое количество контекстов: 497.


In [4]:
print("\n--- Шаг 4: Финализация и сохранение артефактов ---")

final_hybrid_library = {}
for main_type, data in hybrid_library.items():
    final_hybrid_library[main_type] = {
        'brands': sorted(list(data.get('brands', set()))),
        'volumes': sorted(list(data.get('volumes', set()))),
        'percents': sorted(list(data.get('percents', set())))
    }

master_artefacts = {
    "golden_types": golden_types,
    "golden_brands": golden_brands,
    "hybrid_library": final_hybrid_library
}

with open(OUTPUT_LIBRARY_PATH, "wb") as f:
    pickle.dump(master_artefacts, f)

print("Финальные артефакты успешно собраны и сохранены.")
print(f"Путь к файлу: {OUTPUT_LIBRARY_PATH}")
print(f"\nПример данных для категории 'молоко':")
if 'молоко' in final_hybrid_library:
    print(json.dumps(final_hybrid_library['молоко'], ensure_ascii=False, indent=2))
else:
    print("Категория 'молоко' не найдена в качестве основного ключа.")


--- Шаг 4: Финализация и сохранение артефактов ---
Финальные артефакты успешно собраны и сохранены.
Путь к файлу: ../../data/external/hybrid_library.pkl

Пример данных для категории 'молоко':
{
  "brands": [],
  "volumes": [
    "0.45л",
    "0.97л",
    "1 кг",
    "1 л",
    "1.4 кг",
    "1.4 л",
    "1.4кг",
    "1.4л",
    "1.55 л",
    "1.55л",
    "1.947 л",
    "1.947л",
    "100 мл",
    "1000 г",
    "1000г",
    "1030г",
    "10г",
    "125 мл",
    "125г",
    "135 г",
    "135г",
    "1400г",
    "150 г",
    "150г",
    "160 г",
    "160 мл",
    "160г",
    "170 г",
    "170г",
    "180 г",
    "180г",
    "198 мл",
    "1кг",
    "1л",
    "2 л",
    "200 г",
    "200 мл",
    "200г",
    "200мл",
    "205 мл",
    "205г",
    "250 г",
    "250 мл",
    "250г",
    "250мл",
    "260 г",
    "260г",
    "270 мл",
    "270г",
    "2л",
    "3 л",
    "300 г",
    "300 мл",
    "300г",
    "320 г",
    "320 мл",
    "320г",
    "330 мл",
    "330л",
    "350 г",
    "350г